# Background

In this lab session, you'll explore the GloVe model. An **optional** overview is included below to deepen your understanding of how it works in natural language processing.

## GloVe

GloVe (Global Vectors for Word Representation) is a widely used algorithm for generating word embeddings. Rather than predicting surrounding words like word2vec, GloVe takes a different approach — it analyzes how frequently words appear together across an entire corpus. For instance, if "King" and "Man" consistently co-occur, their resulting vectors will be close in the embedding space.

The core idea is to build a **word-context co-occurrence matrix**, where each entry reflects how often a word appears alongside a given context word in the training data. GloVe then applies **matrix factorization** to compress this large matrix into compact, lower-dimensional vectors that preserve the underlying word relationships.

The process unfolds in three steps:

1. **Build the co-occurrence matrix** — Scan the corpus and record how frequently each word-context pair appears together.
2. **Factorize the matrix** — Decompose it into two smaller matrices: a Word-Feature matrix (WF) and a Feature-Context matrix (FC). Both are initialized randomly and refined iteratively using Stochastic Gradient Descent (SGD) to minimize the difference between the approximated matrix (WC') and the original (WC).
3. **Extract the embeddings** — Once training converges, each row of WF becomes the embedding for that word. The embedding size is controlled by F, which you set in advance.

What sets GloVe apart is its ability to combine **global corpus statistics** with **local context signals**, producing embeddings that capture both semantic meaning (what words mean) and syntactic structure (how words relate grammatically).

# Applying pretrained word embeddings 

### Load pre-trained glove model

In [5]:
from torchtext.vocab import GloVe, vocab
import torch
import torch.nn as nn

In [2]:
glove_vector_6B = GloVe(name='6B')

.vector_cache\glove.6B.zip: 862MB [05:27, 2.64MB/s]                                
100%|█████████▉| 399999/400000 [00:58<00:00, 6865.20it/s]


In [4]:
# we can load an advance model also
#glove_840B = GloVe(name='840B', dim=300)

You must continue with the 6B model as it is lighter. You can load different pretrained GloVe models from torch() using ```torch.nn.Embedding.from_pretrained```. 


In [6]:
# load the glove model pretrained weights into a PyTorch embedding layer
embeddings_glove6B = torch.nn.Embedding.from_pretrained(glove_vector_6B.vectors,freeze=True)

In [7]:
embeddings_glove6B

Embedding(400000, 300)

In [13]:
word_to_index = glove_vector_6B.stoi
word_to_index["responsible"]

1442

In [ ]:
#we can get the embedded vector( weight for a word)
embeddings_glove6B.weight[word_to_index["cat"]]

tensor([-0.2935,  0.3325, -0.0474, -0.1225,  0.0720, -0.2341, -0.0624, -0.0037,
        -0.3946, -0.6941,  0.3673, -0.1214, -0.0445, -0.1527,  0.3486,  0.2293,
         0.5436,  0.2521,  0.0980, -0.0873,  0.8706, -0.1221, -0.0798,  0.2871,
        -0.6856, -0.2727,  0.2206, -0.7575,  0.5629,  0.0914, -0.7100, -0.3142,
        -0.5683, -0.2668, -0.6010,  0.2696, -0.1799,  0.1070, -0.5786,  0.3816,
        -0.6713,  0.1093,  0.0794,  0.0224, -0.0811,  0.0112,  0.6709, -0.1909,
        -0.3368, -0.4847, -0.3541, -0.1521,  0.4450,  0.4638,  0.3841,  0.0451,
        -0.5908,  0.2176,  0.3858, -0.4457,  0.0093,  0.4420,  0.0971,  0.3801,
        -0.1188, -0.4272, -0.3101, -0.0251,  0.1269, -0.1347,  0.1198,  0.7625,
         0.2524, -0.2693,  0.0686, -0.1007,  0.0111, -0.1853,  0.4498, -0.5751,
         0.1228, -0.0649,  0.0445, -0.0210, -0.0698, -0.4733, -0.4307,  0.3916,
        -0.0478, -0.9366, -0.5513, -0.1422, -0.1583,  0.1562,  0.0705,  0.1989,
         0.1894, -0.1934, -0.4659, -0.02

In [16]:
words = [
    "fast",
    "slow",
    "bright",
    "dark",
    "shirt",
    "jacket",
    "heavy",
    "light",
    "green",
    "yellow",
    "laugh",
    "cry",
    "sprint",
    "crawl",
    "narrow",
    "wide",
    "smooth",
    "coarse",
    "leader",
    "follower"
]

Create a dictionary of words and their embeddings


In [17]:
embedding_dict_glove6B = {}

for word in words:
    
    # get the embedding value by using vocan as a glove embedding layer
    embedding_vector = embeddings_glove6B.weight[word_to_index[word]]

    if embedding_vector is not None:
        embedding_dict_glove6B[word] = embedding_vector

In [22]:
embedding_dict_glove6B["slow"]

tensor([ 2.4223e-01, -2.1052e-01,  3.1123e-01,  3.6534e-02,  8.2903e-02,
        -6.4373e-01,  5.4033e-01,  4.9111e-01,  3.2448e-02, -1.2789e+00,
        -2.5783e-01,  2.1460e-01, -5.8350e-01, -8.6963e-03,  3.3958e-01,
        -4.2335e-01, -1.4194e-01, -2.8013e-01,  8.9874e-02,  1.5621e-01,
        -2.8058e-01, -4.4189e-02,  1.1011e-01,  6.2551e-01, -3.3791e-01,
         3.7204e-01,  7.8522e-01, -5.7875e-01, -8.9608e-02,  3.8450e-01,
         1.5136e-02,  9.0337e-01,  2.5027e-01,  8.4262e-02, -1.0191e+00,
         7.0148e-01, -2.4310e-01, -3.6418e-02, -4.2877e-01, -1.1958e-01,
        -2.8826e-01,  5.1221e-01, -7.7004e-02, -4.8478e-01,  3.8976e-01,
        -3.1066e-01,  1.5088e-01,  9.2829e-02, -3.7033e-01,  8.8554e-02,
         3.0632e-01, -1.5505e-01, -1.0698e-01, -1.8980e-01, -1.1506e-02,
         2.9179e-01, -1.1534e-01, -3.1570e-01,  2.2013e-01,  6.4830e-01,
         7.0539e-02,  7.8983e-02,  4.5937e-01,  3.0354e-01, -5.3307e-01,
         3.5145e-01,  8.1374e-01,  1.6510e-01,  6.0

Now that you have loaded the pretrained embeddings for the sample words, let's check if the model can capture the similarity of words by finding the distance between words:


In [23]:
def find_similar_words(target_word, embedded_dict, top_k):
    
    target_vector = embedded_dict[target_word]
    similarties = {}
    
    for word, vector in embedded_dict.items():
        if word == target_word:
            continue
        
        similarity = torch.dot(target_vector, vector)/(
            (torch.norm(target_vector))* (torch.norm(vector)))
        
        similarties[word] = similarity
    sorted_words = sorted(similarties.items(),key=lambda x: x[1], reverse=True)

    return sorted_words[:top_k]

In [24]:
target_word = "narrow"
top_k = 2

similar_words = find_similar_words(target_word,embedding_dict_glove6B,top_k)

In [25]:
print("{} most similar words to {}:".format(top_k,target_word) ,similar_words)

2 most similar words to narrow: [('wide', tensor(0.4771)), ('dark', tensor(0.3557))]
